# Step 2 — Hierarchical Gap-Filling (Imputation)

This notebook fills missing discharge values using a three-level hierarchical strategy,
ensuring every station has a continuous daily record for the study period.

## Imputation levels
1. **Short-gap log-interpolation** — Gaps ≤ 15 days are filled by interpolating in log-space.
2. **Donor-based log-regression** — Longer gaps are reconstructed using the best-correlated
   neighbouring station (10 nearest by UTM distance, minimum Pearson r > 0.75 in log-space,
   ≥ 1500 concurrent observations).
3. **Day-of-year climatology** — Any remaining gaps are filled with the station-specific
   DOY median discharge (computed from observed data only).

## Inputs
| File | Description |
|---|---|
| `data/discharge.csv` | Filtered daily discharge from Step 1 |

## Outputs
| File | Description |
|---|---|
| `data/caudales_diarios_imputados_TODO.csv` | Full dataset with all audit columns |
| `data/caudales_diarios_imputados_CORE.csv` | Core columns (Q_imp + flags) |
| `data/caudales_diarios_imputados_AUDIT.csv` | Audit version for reproducibility |

In [ ]:
import numpy as np
import pandas as pd

PATH = "data/discharge.csv"  # o el que uses
df = pd.read_csv(PATH, parse_dates=["date"])

df = df.sort_values(["station_id","date"]).copy()
print(df.columns)
print("Range:", df["date"].min(), "->", df["date"].max())
print("Stations:", df["station_id"].nunique(), "Rows:", len(df))

# Índice temporal para interpolación con method="time"
df = df.set_index("date")

In [ ]:
df.head

In [ ]:
# Gap diagnosis 
def run_lengths(mask: np.ndarray):
    """Return list of lengths of consecutive True runs."""
    m = mask.astype(int)
    if m.sum() == 0:
        return []
    diff = np.diff(np.r_[0, m, 0])
    starts = np.where(diff == 1)[0]
    ends = np.where(diff == -1)[0]
    return (ends - starts).tolist()

rows = []
for sid, g in df.groupby("station_id"):
    q = g["Q"].to_numpy()
    nan_mask = np.isnan(q)
    lens = run_lengths(nan_mask)
    rows.append({
        "station_id": sid,
        "pct_nan": float(np.mean(nan_mask)),
        "nan_days": int(np.sum(nan_mask)),
        "n_gaps": int(len(lens)),
        "max_gap": int(max(lens) if lens else 0),
        "p95_gap": float(np.percentile(lens, 95) if lens else 0),
        "p50_gap": float(np.percentile(lens, 50) if lens else 0),
    })

gap_diag = pd.DataFrame(rows).sort_values(["pct_nan","max_gap"], ascending=False)
gap_diag.describe(percentiles=[.5,.75,.9,.95,.99]), gap_diag.head(10)

## Three-level hierarchical imputation
### Level 1: Short-gap log-interpolation (≤ 15 days)\nShort gaps are filled by interpolation in log-space. All other NaNs are left untouched.

In [ ]:
# Apply to all stations and initialise flags 
df["Q_imp"] = df["Q"].astype(float)
df["imputed_flag"] = False
df["imputed_method"] = ""

# 1) Asegurar numérico
df = df.reset_index() if df.index.name == "date" else df.copy()
df["Q"] = pd.to_numeric(df["Q"], errors="coerce")

# 2) Crear Q_imp nivel 1 (si aún no lo hiciste)
df = df.sort_values(["station_id","date"]).set_index("date")
df["Q_imp"] = df["Q"].astype(float)

# sanity: where Q is not NaN, Q_imp must equal Q
mask = df["Q"].notna()
print("Max abs diff (Q vs Q_imp) en observados:",
      np.nanmax(np.abs(df.loc[mask, "Q"] - df.loc[mask, "Q_imp"])))

print("NaNs Q:", df["Q"].isna().mean(), "NaNs Q_imp:", df["Q_imp"].isna().mean())

# check 5 random observed rows
sample = df.loc[mask].sample(5, random_state=0)[["station_id","Q","Q_imp"]]
sample

In [ ]:
# Apply log-interpolation for gaps <= 15 days and add flags
EPS = 1e-3

def fill_short_gaps_logtime(s: pd.Series, max_gap=15):
    s = s.copy()
    is_na = s.isna()
    grp = (is_na != is_na.shift()).cumsum()
    run_len = is_na.groupby(grp).transform("sum")
    to_fill = is_na & (run_len <= max_gap)

    logq = np.log(s + EPS)
    logq_i = logq.interpolate(method="time", limit=max_gap, limit_direction="both")

    s[to_fill] = np.exp(logq_i[to_fill]) - EPS
    s[s < 0] = 0.0
    return s, to_fill

# Ensure numeric and MultiIndex
df = df.reset_index() if df.index.name == "date" else df.copy()
df["Q"] = pd.to_numeric(df["Q"], errors="coerce")

# MultiIndex únic
df = df.sort_values(["station_id","date"]).set_index(["station_id","date"])

# Inicialise
df["Q_imp"] = df["Q"].astype(float)
df["imputed_flag"] = False
df["imputed_method"] = ""

filled_total = 0

for sid, g in df.groupby(level=0):
    q = g["Q_imp"]
    q_filled, filled_mask = fill_short_gaps_logtime(q.droplevel(0), max_gap=15)
    # q_filled y filled_mask están indexados por date; volvemos a MultiIndex
    idx = pd.MultiIndex.from_product([[sid], q_filled.index], names=["station_id","date"])

    df.loc[idx, "Q_imp"] = q_filled.values
    df.loc[idx[filled_mask.values], "imputed_flag"] = True
    df.loc[idx[filled_mask.values], "imputed_method"] = "log_interp_<=15"
    filled_total += int(filled_mask.sum())

print("NaNs antes:", df["Q"].isna().mean())
print("NaNs después (<=15):", df["Q_imp"].isna().mean())
print("Días imputados (<=15):", filled_total)

df = df.reset_index()  # si prefieres volver a formato plano



### Level 2: Spatial donor (log-regression)

In [ ]:
# Build wide matrix and compute UTM-distance neighbours 
import numpy as np
import pandas as pd

# Ensure correct sort order and types
df = df.copy()
df["Q_imp"] = pd.to_numeric(df["Q_imp"], errors="coerce")
df["Q"] = pd.to_numeric(df["Q"], errors="coerce")
df = df.sort_values(["station_id","date"])

# Station coordinates
coords = (df[["station_id","xutm","yutm"]]
          .drop_duplicates("station_id")
          .dropna(subset=["xutm","yutm"])
          .set_index("station_id"))

station_ids = coords.index.to_numpy()
xy = coords[["xutm","yutm"]].to_numpy()

# Distance matrix
dists = np.sqrt(((xy[:,None,:] - xy[None,:,:])**2).sum(axis=2))
np.fill_diagonal(dists, np.inf)

K_NEIGH = 10
nn_idx = np.argsort(dists, axis=1)[:, :K_NEIGH]
neighbors = {station_ids[i]: station_ids[nn_idx[i]].tolist() for i in range(len(station_ids))}

# Wide matrix: index=date, columns=station_id
wide = df.pivot(index="date", columns="station_id", values="Q_imp").sort_index()
print("Wide shape:", wide.shape)
wide.head()

In [ ]:
# Functions to select the best donor and predict in log-space 
EPS = 1e-3

def fit_log_linear(x, y):
    # y = a + b*x in log-space
    A = np.c_[np.ones_like(x), x]
    coef, *_ = np.linalg.lstsq(A, y, rcond=None)
    return float(coef[0]), float(coef[1])

def pick_best_donor(wide, target_id, donor_ids, min_pairs=1500):
    y = wide[target_id]
    best = None
    best_r = -np.inf
    best_ab = None
    best_n = 0

    for d in donor_ids:
        if d not in wide.columns:
            continue
        x = wide[d]
        xy = pd.concat([x, y], axis=1).dropna()
        n = len(xy)
        if n < min_pairs:
            continue

        lx = np.log(xy.iloc[:,0].values + EPS)
        ly = np.log(xy.iloc[:,1].values + EPS)

        r = np.corrcoef(lx, ly)[0,1]
        if np.isnan(r):
            continue

        if r > best_r:
            a, b = fit_log_linear(lx, ly)
            best = d
            best_r = float(r)
            best_ab = (a, b)
            best_n = n

    return best, best_r, best_ab, best_n

In [ ]:
# Apply donor-based imputation 
R_MIN = 0.75 # minimum acceptable correlation
MIN_PAIRS = 1500  # 60 years daily → 1500 is reasonable and avoids poor fits

wide2 = wide.copy()
donor_info = {}

filled2 = 0

for sid in wide.columns:
    donor, r, ab, n = pick_best_donor(wide2, sid, neighbors.get(sid, []), min_pairs=MIN_PAIRS)
    if donor is None or r < R_MIN:
        continue

    a, b = ab
    donor_info[sid] = {"donor": donor, "r_log": r, "n_pairs": n}

    miss = wide2[sid].isna()
    if miss.sum() == 0:
        continue

    x = wide2[donor]
    can = miss & x.notna()
    if can.sum() == 0:
        continue

    lx = np.log(x[can].values + EPS)
    ly_hat = a + b * lx
    y_hat = np.exp(ly_hat) - EPS
    y_hat[y_hat < 0] = 0.0

    wide2.loc[can, sid] = y_hat
    filled2 += int(can.sum())

print("Días imputados con donante:", filled2)
print("NaNs tras donante:", wide2.isna().mean().mean())
print("Estaciones con donante:", len(donor_info))


In [ ]:
# Convert back to long format and update flags 
# Pivot to long
imp_long = (wide2.stack()
            .rename("Q_imp_new")
            .reset_index())

df2 = df.merge(imp_long, on=["date","station_id"], how="left")

# update only where NaN was filled
mask_new = df2["Q_imp"].isna() & df2["Q_imp_new"].notna()
df2.loc[mask_new, "Q_imp"] = df2.loc[mask_new, "Q_imp_new"]
df2.loc[mask_new, "imputed_flag"] = True
df2.loc[mask_new, "imputed_method"] = "donor_log_reg"

df2 = df2.drop(columns=["Q_imp_new"])

print("NaNs antes nivel 2:", df["Q_imp"].isna().mean())
print("NaNs después nivel 2:", df2["Q_imp"].isna().mean())
print("Días imputados nivel 2:", int(mask_new.sum()))
df2.head()

In [ ]:
# Check how many stations still have NaNs
df2.groupby("station_id")["Q_imp"].apply(lambda s: s.isna().any()).sum()

### Level 3: DOY climatology fallback (station-specific median)

In [ ]:
# Compute DOY climatology from observed data only
# Ensure sort order
df2 = df2.sort_values(["station_id","date"]).copy()

# DOY (handling leap years)
df2["doy"] = df2["date"].dt.dayofyear
df2.loc[df2["doy"] == 366, "doy"] = 365

# DOY climatology per station (median)
clim_doy = (
    df2[df2["Q"].notna()]
    .groupby(["station_id","doy"])["Q"]
    .median()
    .reset_index()
    .rename(columns={"Q": "Q_clim_doy"})
)

clim_doy.head()

In [ ]:
# Fill remaining NaNs with DOY climatology
df3 = df2.merge(clim_doy, on=["station_id","doy"], how="left")

fallback_mask = df3["Q_imp"].isna() & df3["Q_clim_doy"].notna()

df3.loc[fallback_mask, "Q_imp"] = df3.loc[fallback_mask, "Q_clim_doy"]
df3.loc[fallback_mask, "imputed_flag"] = True
df3.loc[fallback_mask, "imputed_method"] = "doy_climatology"

print("NaNs finales:", df3["Q_imp"].isna().sum())
print("Días imputados nivel 3:", int(fallback_mask.sum()))
df3.head()

In [ ]:
# Final traceability check 
df3["imputed_method"].value_counts()

## Final checks 

In [ ]:
(df3["Q_imp"] < 0).sum(), df3["Q_imp"].describe(percentiles=[0.001, 0.999])

In [ ]:
# Check where DOY climatology was applied (per station)
df3["is_clim"] = df3["imputed_method"] == "doy_climatology"

(df3.groupby("station_id")["is_clim"].mean()
     .sort_values(ascending=False)
     .head(10))

In [ ]:
# Save imputation flag column
df3["is_imputed"] = df3["imputed_flag"].astype(int)

In [ ]:
# Final summary and save 
# FINAL IMPUTATION SUMMARY
summary = {
    "total_rows": len(df3),
    "pct_observed": (df3["imputed_flag"] == False).mean(),
    "pct_log_interp": (df3["imputed_method"] == "log_interp_<=15").mean(),
    "pct_donor": (df3["imputed_method"] == "donor_log_reg").mean(),
    "pct_climatology": (df3["imputed_method"] == "doy_climatology").mean(),
}

summary

In [ ]:
df3.to_csv(
    "data/caudales_diarios_imputados_TODO.csv",
    index=False,
    date_format="%Y-%m-%d"
)

In [ ]:
df3.head()

In [ ]:
# Core version (for modelling and plots)
df_core = df3[[
    "station_id", "date", "Q_imp", "is_imputed", "imputed_method"
]].copy()

# Audit version (for reproducibility and QA)
df_audit = df3[[
    "station_id", "date", "Q", "Q_imp", "xutm", "yutm",
    "imputed_method", "is_imputed"
]].copy()

df_core.to_csv("data/caudales_diarios_imputados_CORE.csv", index=False, date_format="%Y-%m-%d")
df_audit.to_csv("data/caudales_diarios_imputados_AUDIT.csv", index=False, date_format="%Y-%m-%d")